# Seq2Seq 모델 Q&A Chatbot 구현

1. QnA 데이터셋을 찾아서 처리해서 준비한다. (전처리 전반)
2. Encoder, Decoder, Seq2Seq(Encoder+Decoder) 모델을 만든다.
3. 1에서 준비한 데이터로 2에서 만든 모델을 학습시킨다.
4. Chatbot을 만든다.(모델 추론 + while문)

In [2]:
import pandas as pd
from tokenizers import BertWordPieceTokenizer
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
df = pd.read_json('./data/raw/kor_conversation/감성대화말뭉치(최종데이터)_Training.json')

In [4]:
contents = []

contents = [line['content'] for line in df['talk']]
print(len(contents), contents[:5])

51628 [{'HS01': '일은 왜 해도 해도 끝이 없을까? 화가 난다.', 'SS01': '많이 힘드시겠어요. 주위에 의논할 상대가 있나요?', 'HS02': '그냥 내가 해결하는 게 나아. 남들한테 부담 주고 싶지도 않고.', 'SS02': '혼자 해결하기로 했군요. 혼자서 해결하기 힘들면 주위에 의논할 사람을 찾아보세요. ', 'HS03': '', 'SS03': ''}, {'HS01': '이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나.', 'SS01': '급여가 줄어 속상하시겠어요. 월급이 줄어든 것을 어떻게 보완하실 건가요?', 'HS02': '최대한 지출을 억제해야겠어. 월급이 줄어들었으니 고정지출을 줄일 수밖에 없을 것 같아.', 'SS02': '월급이 줄어든 만큼 소비를 줄일 계획이군요.', 'HS03': '', 'SS03': ''}, {'HS01': '회사에 신입이 들어왔는데 말투가 거슬려. 그런 애를 매일 봐야 한다고 생각하니까 스트레스 받아. ', 'SS01': '회사 동료 때문에 스트레스를 많이 받는 것 같아요. 문제 해결을 위해 어떤 노력을 하면 좋을까요?', 'HS02': '잘 안 맞는 사람이랑 억지로 잘 지내는 것보단 조금은 거리를 두고 예의를 갖춰서 대하는 게 덜 스트레스받을 것 같아.', 'SS02': '스트레스받지 않기 위해선 인간관계에 있어 약간의 거리를 두는 게 좋겠군요.', 'HS03': '', 'SS03': ''}, {'HS01': '직장에서 막내라는 이유로 나에게만 온갖 심부름을 시켜. 일도 많은 데 정말 분하고 섭섭해.', 'SS01': '관련 없는 심부름을 모두 하게 되어서 노여우시군요. 어떤 것이 상황을 나아질 수 있게 도움을 줄까요?', 'HS02': '직장 사람들과 솔직하게 이야기해보고 싶어. 일하는 데에 방해된다고.', 'SS02': '직장 사람들과 이야기를 해 보겠다고 결심하셨군요.', 'HS03': '', 'SS03': ''}, {'HS01': '얼마 전 입사한 신입사

In [5]:
conversation_df = pd.DataFrame(contents)
conversation_df

,HS01,SS01,HS02,SS02,HS03,SS03
0,일은 왜 해도 해도 끝이 없을까? 화가 난다.,많이 힘드시겠어요. 주위에 의논할 상대가 있나요?,그냥 내가 해결하는 게 나아. 남들한테 부담 주고 싶지도 않고.,혼자 해결하기로 했군요. 혼자서 해결하기 힘들면 주위에 의논할 사람을 찾아보세요.,,
1,이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나.,급여가 줄어 속상하시겠어요. 월급이 줄어든 것을 어떻게 보완하실 건가요?,최대한 지출을 억제해야겠어. 월급이 줄어들었으니 고정지출을 줄일 수밖에 없을 것 같아.,월급이 줄어든 만큼 소비를 줄일 계획이군요.,,
2,회사에 신입이 들어왔는데 말투가 거슬려. 그런 애를 매일 봐야 한다고 생각하니까 스...,회사 동료 때문에 스트레스를 많이 받는 것 같아요. 문제 해결을 위해 어떤 노력을 ...,잘 안 맞는 사람이랑 억지로 잘 지내는 것보단 조금은 거리를 두고 예의를 갖춰서 대...,스트레스받지 않기 위해선 인간관계에 있어 약간의 거리를 두는 게 좋겠군요.,,
3,직장에서 막내라는 이유로 나에게만 온갖 심부름을 시켜. 일도 많은 데 정말 분하고 ...,관련 없는 심부름을 모두 하게 되어서 노여우시군요. 어떤 것이 상황을 나아질 수 있...,직장 사람들과 솔직하게 이야기해보고 싶어. 일하는 데에 방해된다고.,직장 사람들과 이야기를 해 보겠다고 결심하셨군요.,,
4,얼마 전 입사한 신입사원이 나를 무시하는 것 같아서 너무 화가 나.,무시하는 것 같은 태도에 화가 나셨군요. 상대방의 어떤 행동이 그런 감정을 유발하는...,상사인 나에게 먼저 인사하지 않아서 매일 내가 먼저 인사한다고!,항상 먼저 인사하게 되어 화가 나셨군요. 어떻게 하면 신입사원에게 화났음을 표현할 ...,,
...,...,...,...,...,...,...
51623,나이가 먹고 이제 돈도 못 벌어 오니까 어떻게 살아가야 할지 막막해. 능력도 없고.,경제적인 문제 때문에 막막하시군요. 마음이 편치 않으시겠어요.,아무것도 할 수 없는 내가 무가치하게 느껴지고 실망스러워.,지금 할 수 있는 가장 합리적인 행동은 무엇인가요?,노년층을 위한 경제적 지원이나 부업 같은 것도 알아보아야겠어.,좋은 결과 얻으시길 바랄게요.
51624,몸이 많이 약해졌나 봐. 이제 전과 같이 일하지 못할 것 같아 너무 짜증 나.,건강에 대한 어려움 때문에 기분이 좋지 않으시군요. 속상하시겠어요.,마음 같아서는 다 할 수 있는 일인데 이젠 몸이 안 따라와 주니 화만 나.,어떻게 하면 지금의 기분을 나아지게 할 수 있을까요?,남편과 함께 게이트볼이나 치러 가야겠어. 그럼 기분이 나아질 것 같아.,남편과 함께하는 좋은 외출 시간 되시길 바랄게요.
51625,이제 어떻게 해야 할지 모르겠어. 남편도 그렇고 노후 준비도 안 되어서 미래가 걱정돼.,노후 준비에 대한 어려움 때문에 걱정이 많으시겠어요.,주변 사람들은 다 노후 준비도 잘해두었던데 난 어떻게 해야 할지 모르겠어. 막막하기...,지금의 상황에서 할 수 있는 가장 좋은 행동이 무엇일까요?,남편과 함께 실버 일자리나 노년층을 위한 국가 지원에 대해 자세히 알아보아야겠어.,좋은 정보 많이 얻으셔서 걱정을 좀 덜으셨으면 좋겠어요.
51626,몇십 년을 함께 살았던 남편과 이혼했어. 그동안의 세월에 배신감을 느끼고 너무 화가 나.,가족과의 문제 때문에 속상하시겠어요.,이제 할 수 있는 일도 없고 이렇게 힘들게 사는 게 불만스럽기만 해.,지금의 감정을 나아지게 할 수 있는 어떤 방법이 있을까요?,함께 친하게 지내던 동네 언니 동생들과 빈자리를 조금이나마 채울까 해.,지인분들과 좋은 시간 보내셨으면 좋겠어요.


In [6]:
len(conversation_df[conversation_df['HS03'] == ''])

8935

In [ ]:
q1 = conversation_df['HS01'].to_list()
q2 = conversation_df['SS01'].to_list()
q3 = conversation_df['HS02'].to_list()
q4 = conversation_df[conversation_df['HS03'] != '']['SS02'].to_list()
q5 = conversation_df[conversation_df['SS03'] != '']['HS03'].to_list()

encoder_inputs = q1+q2+q3+q4+q5
encoder_inputs = [sent for sent in encoder_inputs]

['[SOS] 일은 왜 해도 해도 끝이 없을까? 화가 난다.', '[SOS] 이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나.', '[SOS] 회사에 신입이 들어왔는데 말투가 거슬려. 그런 애를 매일 봐야 한다고 생각하니까 스트레스 받아. ']


In [34]:
a1 = conversation_df['SS01'].to_list()
a2 = conversation_df['HS02'].to_list()
a3 = conversation_df['SS02'].to_list()
a4 = conversation_df[conversation_df['HS03'] != '']['HS03'].to_list()
a5 = conversation_df[conversation_df['SS03'] != '']['SS03'].to_list()

targets = a1+a2+a3+a4+a5
targets = [sent + ' [EOS]' for sent in targets]
print(targets[:3])

['많이 힘드시겠어요. 주위에 의논할 상대가 있나요? [EOS]', '급여가 줄어 속상하시겠어요. 월급이 줄어든 것을 어떻게 보완하실 건가요? [EOS]', '회사 동료 때문에 스트레스를 많이 받는 것 같아요. 문제 해결을 위해 어떤 노력을 하면 좋을까요? [EOS]']


In [ ]:
from tqdm import tqdm
from konlpy.tag import Okt

def tokenizing(sentences):
    okt = Okt()
    return [okt.morphs(sent) for sent in tqdm(sentences)]

In [ ]:
# import pandas as pd
# from tqdm import tqdm
# from konlpy.tag import Okt
# from concurrent.futures import ProcessPoolExecutor

# all_questions = [q1, q2, q3, q4, q5, q6]
    

# with ProcessPoolExecutor(max_workers=6) as executor:
#     # map을 사용하면 입력 순서대로 결과가 반환됩니다.
#     results = list(tqdm(executor.map(tokenizing, all_questions), total=len(all_questions)))

#     # 3. 결과 다시 할당
#     tokenized_q1, tokenized_q2, tokenized_q3, tokenized_q4, tokenized_q5, tokenized_q6 = results

  0%|          | 0/6 [00:00<?, ?it/s]


In [59]:
tokenizer = BertWordPieceTokenizer(lowercase=False, strip_accents=False)
vocab_size = 1000000

tokenizer.train(
    files= ['./data/raw/kor_conversation/감성대화말뭉치(최종데이터)_Training.json'],
    vocab_size=vocab_size,
    min_frequency=5,
    show_progress=True,
    special_tokens=['[PAD]', '[UNK]', '[MASK]', '[SOS]', '[EOS]']
)

MAX_LEN = 55

In [60]:
q1 = conversation_df['HS01'].to_list()
q2 = conversation_df['SS01'].to_list()
q3 = conversation_df['HS02'].to_list()
q4 = conversation_df[conversation_df['HS03'] != '']['SS02'].to_list()
q5 = conversation_df[conversation_df['SS03'] != '']['HS03'].to_list()

encoder_inputs = q1+q2+q3+q4+q5
encoder_inputs = [sent for sent in encoder_inputs]

encoder_inputs = [tokenizer.encode(sent).ids for sent in encoder_inputs]
encoder_input_max_len = max(len(s) for s in encoder_inputs)
print(f'{encoder_input_max_len = }')

encoder_inputs = pad_sequences(encoder_inputs, maxlen=MAX_LEN, padding='pre')

encoder_input_max_len = 53


In [61]:
a1 = conversation_df['SS01'].to_list()
a2 = conversation_df['HS02'].to_list()
a3 = conversation_df['SS02'].to_list()
a4 = conversation_df[conversation_df['HS03'] != '']['HS03'].to_list()
a5 = conversation_df[conversation_df['SS03'] != '']['SS03'].to_list()

targets = a1+a2+a3+a4+a5
decoder_inputs = ['[SOS] ' + sent for sent in targets]
decoder_targets = [sent + ' [EOS]' for sent in targets]


decoder_inputs = [tokenizer.encode(sent).ids for sent in decoder_inputs]
decoder_targets = [tokenizer.encode(sent).ids for sent in decoder_targets]

decoder_targets_max_len = max(len(s) for s in decoder_targets)

print(f'{decoder_targets_max_len = }')

decoder_inputs = pad_sequences(decoder_inputs, maxlen=MAX_LEN, padding='pre')
decoder_targets = pad_sequences(decoder_targets, maxlen=MAX_LEN, padding='post')

decoder_targets_max_len = 52


In [ ]:
class MMTDataset(Dataset):
  def __init__(self, encoder_inputs, decoder_inputs, decoder_targets):
    super().__init__()
    self.encoder_inputs = encoder_inputs
    self.decoder_inputs = decoder_inputs
    self.decoder_targets = decoder_targets

  def __len__(self):
    return len(self.encoder_inputs)

  def __getitem__(self, index):
    return (
        torch.tensor(self.encoder_inputs[index], dtype=torch.long),
        torch.tensor(self.decoder_inputs[index], dtype=torch.long),
        torch.tensor(self.decoder_targets[index], dtype=torch.long),
    )

In [ ]:
train_index, val_index = train_test_split(range(len(encoder_inputs)), random_state=0)
print(len(train_index), len(val_index))

train_dataset = MMTDataset(
    encoder_inputs[train_index],
    decoder_inputs[train_index],
    decoder_targets[train_index]
)
val_dataset = MMTDataset(
    encoder_inputs[val_index],
    decoder_inputs[val_index],
    decoder_targets[val_index]
)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [ ]:
class Encoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, latent_dim, embedding_matrix=None):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    if embedding_matrix is not None:
      self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
    self.lstm = nn.LSTM(embedding_dim, latent_dim, batch_first=True)

  def forward(self, X):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X)
    return h_s, c_s

In [ ]:
class Decoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, latent_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    self.lstm = nn.LSTM(embedding_dim, latent_dim, batch_first=True)
    self.fc = nn.Linear(latent_dim, vocab_size)

  def forward(self, X, hidden, cell):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X, (hidden, cell))
    logits = self.fc(output)
    return logits, h_s, c_s

In [ ]:
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder

  def forward(self, source, target):
    h_s, c_s = self.encoder(source)
    output, h_s, c_s = self.decoder(target, h_s, c_s)
    return output

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(eng_num_words, EMBEDDING_DIM, LATENT_DIM, embedding_matrix)
decoder = Decoder(kor_num_words, EMBEDDING_DIM, LATENT_DIM)
model = Seq2Seq(encoder, decoder).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.AdamW(model.parameters(), lr=0.001)

epochs = 100

train_losses, train_accs, val_losses, val_accs = [], [], [], []

for epoch in range(epochs):
  model.train()
  train_loss, train_correct, train_tokens = 0, 0, 0

  for enc_inputs, dec_inputs, dec_targets in train_loader:
    enc_inputs = enc_inputs.to(device)
    dec_inputs = dec_inputs.to(device)
    dec_targets = dec_targets.to(device)

    optimizer.zero_grad()

    # teacher forcing
    output = model(enc_inputs, dec_inputs)
    output = output.view(-1, output.size(-1))
    dec_targets = dec_targets.view(-1)

    loss = criterion(output, dec_targets)
    loss.backward()
    optimizer.step()

    preds = output.argmax(dim=-1)
    train_loss += loss.detach().cpu().item()
    mask = dec_targets != 0
    correct = (preds == dec_targets) & mask
    train_correct += correct.sum().detach().cpu().item()
    train_tokens += mask.sum().detach().cpu().item()

  train_loss /= len(train_loader)
  train_acc = train_correct / train_tokens
  train_losses.append(train_loss)
  train_accs.append(train_acc)

  model.eval()
  with torch.no_grad():
    val_loss, val_correct, val_tokens = 0, 0, 0

    for enc_inputs, dec_inputs, dec_targets in val_loader:
      enc_inputs = enc_inputs.to(device)
      dec_inputs = dec_inputs.to(device)
      dec_targets = dec_targets.to(device)

      output = model(enc_inputs, dec_inputs)
      output = output.view(-1, output.size(-1))
      dec_targets = dec_targets.view(-1)

      loss = criterion(output, dec_targets)

      preds = output.argmax(dim=-1)
      val_loss += loss.detach().cpu().item()
      mask = dec_targets != 0
      correct = (preds == dec_targets) & mask
      val_correct += correct.sum().detach().cpu().item()
      val_tokens += mask.sum().detach().cpu().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_tokens
    val_losses.append(val_loss)
    val_accs.append(val_acc)

  print(f'Epoch {epoch+1}/{epochs} TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}')